In [15]:
# --- 1. Selenium으로 '목록 페이지'에서 모든 청원 ID 수집 (전체 페이지) ---

import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import pandas as pd
import requests 
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException

# -----------------------------------------------------------------
# 헬퍼 함수 (수정 없음)
# -----------------------------------------------------------------
def get_current_active_page(driver):
    try:
        element = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.cs-paging a.active"))
        )
        return element.text
    except Exception as e:
        print(f"  [경고] 현재 페이지 번호를 찾는 데 실패: {e}")
        return "-1"

def wait_for_list_to_refresh(driver, old_pid):
    print("  새 목록이 로드될 때까지 대기 중...")
    try:
        WebDriverWait(driver, 10).until(
            lambda d: old_pid not in d.find_element(By.CSS_SELECTOR, "ul.list_card li a").get_attribute('onclick')
        )
        print("  새 목록 로드 확인.")
    except (TimeoutException, StaleElementReferenceException): # Stale 에러도 여기서 잡을 수 있음
        print("  [알림] 목록 새로고침 감지됨 (정상).")
    except Exception as e:
        print(f"  [경고] 목록 새로고침 대기 중 오류: {e}")


# -----------------------------------------------------------------
# 1. 셀레니움 드라이버 시작
# -----------------------------------------------------------------
service = Service(executable_path='./chromedriver.exe')
driver = webdriver.Chrome(service=service)
driver.implicitly_wait(5) 

list_url = "https://www.cheongwon.go.kr/portal/petition/open/view"
print(f"'{list_url}' 접속 중...")
driver.get(list_url)

all_petition_ids = []

for page_group in range(50): # 1~10, 11~20, 21~30 ... (넉넉하게 50번)
    
    current_page_number = get_current_active_page(driver)
    if current_page_number == "-1":
        print(" [오류] 현재 페이지 번호를 찾을 수 없어 중단합니다.")
        break
        
    print(f"\n--- 페이지 그룹 {page_group + 1} (페이지 {current_page_number}~ ) 스크래핑 시작 ---")
    
    # [!!! LOGIC CHANGED !!!]
    # 'page_buttons_in_group' 리스트를 미리 만들지 않습니다.
    # 대신, 10번의 페이지 클릭을 시도합니다. (1번은 현재 페이지, 9번은 다음 페이지들)
    
    for i in range(10): # 1페이지 + 9페이지 = 총 10페이지
        try:
            # 1. 현재 페이지 번호(예: '361')를 다시 확인합니다.
            current_page_number = get_current_active_page(driver)
            
            # i=0 (첫 번째 루프)가 아니면, 다음 페이지 버튼을 찾아 클릭합니다.
            if i > 0:
                page_to_click_str = str(int(current_page_number) + 1)
                print(f"  [ {page_to_click_str} ] 페이지로 이동...")

                # [!!! THE FIX !!!]
                # 버튼을 '미리' 찾아놓는 게 아니라, '지금' 찾습니다.
                # (By.LINK_TEXT는 '362'라는 텍스트를 가진 <a> 태그를 찾으라는 뜻)
                button = driver.find_element(By.LINK_TEXT, page_to_click_str)
                
                driver.execute_script("arguments[0].scrollIntoView(true);", button)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", button)
                
                # '362'번 버튼이 'active'가 될 때까지 대기
                WebDriverWait(driver, 10).until(
                    EC.text_to_be_present_in_element(
                        (By.CSS_SELECTOR, "div.cs-paging a.active"), 
                        page_to_click_str
                    )
                )
                current_page_number = page_to_click_str # 현재 페이지 번호 갱신
            
            # 2. 현재 페이지(i=0이면 '361', i=1이면 '362')를 긁습니다.
            print(f"  [ {current_page_number} ] 페이지 긁는 중...")
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul.list_card span.subject")))
            
            links = driver.find_elements(By.CSS_SELECTOR, "ul.list_card li a")
            page_ids_found = 0
            for link in links:
                pid = link.get_attribute('onclick').split("'")[1]
                if pid not in all_petition_ids:
                    all_petition_ids.append(pid)
                    page_ids_found += 1
            print(f"  [ {current_page_number} ] 페이지 완료. {page_ids_found}개 추가. (총 {len(all_petition_ids)}개)")

        except NoSuchElementException:
            # '369' 다음 '370'을 찾으려다 실패한 경우입니다. (페이지 그룹의 끝)
            print(f"  페이지 그룹의 마지막 페이지입니다. 다음 10개로 넘어갑니다.")
            break # 10번 돌기 전에 for i in range(10) 루프 탈출
        except Exception as e:
            print(f"  [오류] {current_page_number} 페이지 작업 중 오류: {e}")
            # 이 페이지는 건너뛰고 다음 페이지(i+1) 시도

    # -----------------------------------------------------------------
    # 4. '다음 10개(>)' 버튼을 눌러 다음 그룹(예: 11~20)으로 이동합니다.
    # -----------------------------------------------------------------
    try:
        print("\n  '다음 10개(>)' 버튼을 찾아 클릭합니다...")
        next_group_button = driver.find_element(By.CSS_SELECTOR, "a.cs-paging__next")
        
        driver.execute_script("arguments[0].scrollIntoView(true);", next_group_button)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", next_group_button)
        
        # '다음'을 눌렀으니, 다음 페이지 그룹의 첫 번째 번호(예: '11')가 활성화될 때까지 대기
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.cs-paging a.active"))
        )
        current_page_number = get_current_active_page(driver) # '11'
        print(f"  성공. 다음 페이지 그룹 [ {current_page_number} ] 로드 완료.")

    except NoSuchElementException:
        print("\n'다음 10개(>)' 버튼을 더 이상 찾을 수 없습니다. (NoSuchElementException)")
        print("모든 페이지 수집을 완료했습니다.")
        break # for page_group 루프를 탈출합니다. (최종 종료)
    except Exception as e:
        print(f"\n'다음 10개(>)' 버튼 클릭 중 오류 발생: {e}")
        print("페이지 순회를 중단합니다.")
        break

# -----------------------------------------------------------------
# 5. 모든 작업 완료 후 드라이버 종료
# -----------------------------------------------------------------
driver.quit()
print(f"\n--- 1단계 ID 수집 완료 ---")
print(f"총 {len(all_petition_ids)}개의 고유한 청원 ID를 수집했습니다.")

# --- 2. '상세 페이지' 스크래핑 함수 (버전 1 재활용) ---

def scrape_detail_page(pid):
    """청원 ID를 받아서 상세 페이지를 스크래핑하고 dict로 반환"""
    
    detail_url = f"https://www.cheongwon.go.kr/portal/petition/open/viewdetail/{pid}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        response = requests.get(detail_url, headers=headers)
        if response.status_code != 200:
            print(f"  [실패] {pid} 페이지 응답 없음 (코드: {response.status_code})")
            return None
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        data = {
            'id': pid,
            'url': detail_url,
            'title': soup.select_one('div.subject#titles').get_text(strip=True),
            'department': soup.select_one('div.category > span').get_text(strip=True).replace('처리기관:', '').strip(),
            'date_range': soup.select_one('div.pet-doc-extra h3:first-of-type span.brd').get_text(strip=True),
            'views': soup.select_one('div.link > div[style*="font-size:17px"]').get_text(strip=True).replace('조회 수', '').strip(),
            'content': soup.select_one('div.pet-doc__cont#ptnCns').get_text(separator='\n', strip=True)
        }
        print(f"  [성공] {pid} 스크래핑 완료: {data['title'][:20]}...")
        return data
        
    except Exception as e:
        print(f"  [오류] {pid} 스크래핑 중 예외 발생: {e}")
        return None

# --- 3. 모든 ID를 순회하며 스크래핑 실행 ---

all_petitions_data = []
# [!!! 중요 !!!] 변수명이 all_petition_ids 로 통일되었습니다.
for pid in all_petition_ids: 
    result = scrape_detail_page(pid)
    if result:
        all_petitions_data.append(result)
    time.sleep(0.5) # 서버에 부담을 주지 않기 위해 0.5초 휴식

# --- 4. 최종 결과를 DataFrame으로 변환 후 CSV 저장 ---

if all_petitions_data:
    df = pd.DataFrame(all_petitions_data)
    
    # CSV 파일로 저장 (한글 깨짐 방지: utf-8-sig)
    df.to_csv("청원24_데이터.csv", index=False, encoding='utf-8-sig')
    
    print("\n--- 작업 완료! ---")
    print(df.head())
    print("\n'청원24_데이터.csv' 파일이 저장되었습니다.")
else:
    print("스크래핑된 데이터가 없습니다.")

'https://www.cheongwon.go.kr/portal/petition/open/view' 접속 중...

--- 페이지 그룹 1 (페이지 1~ ) 스크래핑 시작 ---
  [ 1 ] 페이지 긁는 중...
  [ 1 ] 페이지 완료. 12개 추가. (총 12개)
  [ 2 ] 페이지로 이동...
  [ 2 ] 페이지 긁는 중...
  [ 2 ] 페이지 완료. 12개 추가. (총 24개)
  [ 3 ] 페이지로 이동...
  [ 3 ] 페이지 긁는 중...
  [ 3 ] 페이지 완료. 12개 추가. (총 36개)
  [ 4 ] 페이지로 이동...
  [ 4 ] 페이지 긁는 중...
  [ 4 ] 페이지 완료. 12개 추가. (총 48개)
  [ 5 ] 페이지로 이동...
  [ 5 ] 페이지 긁는 중...
  [ 5 ] 페이지 완료. 12개 추가. (총 60개)
  [ 6 ] 페이지로 이동...
  [ 6 ] 페이지 긁는 중...
  [ 6 ] 페이지 완료. 12개 추가. (총 72개)
  [ 7 ] 페이지로 이동...
  [ 7 ] 페이지 긁는 중...
  [ 7 ] 페이지 완료. 12개 추가. (총 84개)
  [ 8 ] 페이지로 이동...
  [ 8 ] 페이지 긁는 중...
  [ 8 ] 페이지 완료. 12개 추가. (총 96개)
  [ 9 ] 페이지로 이동...
  [ 9 ] 페이지 긁는 중...
  [ 9 ] 페이지 완료. 12개 추가. (총 108개)
  [ 10 ] 페이지로 이동...
  [ 10 ] 페이지 긁는 중...
  [ 10 ] 페이지 완료. 12개 추가. (총 120개)

  '다음 10개(>)' 버튼을 찾아 클릭합니다...
  성공. 다음 페이지 그룹 [ 11 ] 로드 완료.

--- 페이지 그룹 2 (페이지 11~ ) 스크래핑 시작 ---
  [ 11 ] 페이지 긁는 중...
  [ 11 ] 페이지 완료. 12개 추가. (총 132개)
  [ 12 ] 페이지로 이동...
  [ 12 ] 페이지 긁는 중...
  [ 12 ] 페